## 230968174
## Smeet Pancholi
### week 10
### 6th april 2026

## Exercise 1: Q-Learning (Grid Robot)

In [3]:

import numpy as np
import random

In [4]:
# creating a simple 5x5 grid environment with obstacles and goal
grid_size = 5
goal = (4, 4)
obstacles = [(1,1), (2,2), (3,1)]

actions = ['up', 'down', 'left', 'right']

In [5]:
# initializing Q-table with zeros for all states and actions
Q = {}
for i in range(grid_size):
    for j in range(grid_size):
        Q[(i,j)] = {a: 0 for a in actions}

In [6]:
# defining how agent moves in the grid
def move(state, action):
    x, y = state
    if action == 'up': x -= 1
    elif action == 'down': x += 1
    elif action == 'left': y -= 1
    elif action == 'right': y += 1
    
    new_state = (x, y)
    
    if x < 0 or x >= grid_size or y < 0 or y >= grid_size or new_state in obstacles:
        return state
    return new_state

# defining reward system
def reward(state):
    if state == goal:
        return 10
    return -1

In [7]:
# training agent using Q-learning algorithm
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

for _ in range(episodes):
    state = (0,0)
    
    while state != goal:
        if random.uniform(0,1) < epsilon:
            action = random.choice(actions)
        else:
            action = max(Q[state], key=Q[state].get)
        
        next_state = move(state, action)
        r = reward(next_state)
        
        Q[state][action] += alpha * (r + gamma * max(Q[next_state].values()) - Q[state][action])
        
        state = next_state

In [8]:
# testing learned path from start to goal
state = (0,0)
path = [state]

while state != goal:
    action = max(Q[state], key=Q[state].get)
    state = move(state, action)
    path.append(state)

print("Path:", path)

Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (3, 4), (4, 4)]


## Exercise 2: Multi-Armed Bandit

In [10]:

import numpy as np

In [11]:
# defining bandit with random probabilities
n_arms = 5
true_probs = np.random.rand(n_arms)
counts = np.zeros(n_arms)
values = np.zeros(n_arms)

In [12]:
# running epsilon greedy strategy
epsilon = 0.1
steps = 1000

for t in range(steps):
    if np.random.rand() < epsilon:
        arm = np.random.randint(n_arms)
    else:
        arm = np.argmax(values)
    
    reward = 1 if np.random.rand() < true_probs[arm] else 0
    
    counts[arm] += 1
    values[arm] += (reward - values[arm]) / counts[arm]

In [13]:
# printing results of bandit experiment
print("Estimated values:", values)
print("Times selected:", counts)

Estimated values: [0.73142857 0.15789474 0.34615385 0.         0.81967213]
Times selected: [875.  19.  26.  19.  61.]


Exercise 3: CartPole (DQN

In [15]:
!pip install gym

In [16]:
!pip install gymnasium

In [17]:

import gymnasium as gym

In [18]:
# importing gym and tensorflow for DQN
import gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [19]:
# creating cartpole environment
env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

In [20]:
# building a simple neural network model
model = tf.keras.Sequential([
    layers.Dense(24, activation='relu', input_shape=(state_size,)),
    layers.Dense(24, activation='relu'),
    layers.Dense(action_size, activation='linear')
])

model.compile(optimizer='adam', loss='mse')

C:\Users\Smeet Pancholi\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
# training loop for DQN (basic version)
episodes = 100
gamma = 0.95
epsilon = 1.0

for e in range(episodes):
    state = env.reset()[0]
    state = np.reshape(state, [1, state_size])
    
    for time in range(200):
        if np.random.rand() <= epsilon:
            action = np.random.choice(action_size)
        else:
            action = np.argmax(model.predict(state, verbose=0))
        
        next_state, reward, done, _, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_size])
        
        target = reward
        if not done:
            target += gamma * np.max(model.predict(next_state, verbose=0))
        
        target_f = model.predict(state, verbose=0)
        target_f[0][action] = target
        
        model.fit(state, target_f, epochs=1, verbose=0)
        state = next_state
        
        if done:
            break
    
    epsilon = max(0.01, epsilon * 0.995)

C:\Users\Smeet Pancholi\anaconda3\Lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


## Exercise 4: Traffic Signal (Q-learning)

In [23]:
# simple traffic simulation using Q-learning
states = [0,1,2,3]  # congestion levels
actions = [0,1]     # 0 = road A, 1 = road B

Q = np.zeros((4,2))

alpha = 0.1
gamma = 0.9

for _ in range(500):
    state = np.random.choice(states)
    
    for _ in range(10):
        action = np.argmax(Q[state])
        
        next_state = np.random.choice(states)
        reward = -next_state
        
        Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
        
        state = next_state

print(Q)

[[-14.76834298 -15.69186388]
 [-15.69473204 -14.3959466 ]
 [-15.74586711 -14.88980874]
 [-14.67576181 -15.41031713]]


### Exercise 5: Bayes Theorem

In [25]:
# computing disease probability using bayes theorem
P_disease = 0.01
P_symptom_given_disease = 0.9
P_symptom_given_no_disease = 0.2

P_symptom = (P_symptom_given_disease * P_disease) + (P_symptom_given_no_disease * (1 - P_disease))

P_disease_given_symptom = (P_symptom_given_disease * P_disease) / P_symptom

print("Probability of disease:", P_disease_given_symptom)

# simple classification based on threshold
if P_disease_given_symptom > 0.5:
    print("Disease Present")
else:
    print("No Disease")

Probability of disease: 0.043478260869565216
No Disease
